# Stage 3.5 control — untrained base model with supplied mapping

This is the exact declared-mapping greenhouse-fan control, evaluated on the untrained Qwen2.5-3B-Instruct base with no adapter loaded. It is read-only, resumable, and contains no training or Stage 4 composition.


In [1]:
%pip install -q transformers==5.13.1 peft==0.19.1 bitsandbytes==0.50.0 accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 65.3 MB/s eta 0:00:00:00:0100:01


In [2]:
import hashlib, json, os, random
from collections import Counter
from pathlib import Path
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import numpy as np
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive',force_remount=False)
if not torch.cuda.is_available(): raise RuntimeError('A Colab GPU is required.')
SEED=20260811
MODEL_NAME='Qwen/Qwen2.5-3B-Instruct'
OUTPUT_DIR=Path('/content/drive/MyDrive/AISI/checkpoints/stage35-base-declared-mapping-control-v1')
PROGRESS=OUTPUT_DIR/'rollouts.json'; REPORT=OUTPUT_DIR/'report.json'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print({'gpu':torch.cuda.get_device_name(0),'model':MODEL_NAME,
       'adapter_loaded':False,'training':False,'stage4_blocked':True})


Mounted at /content/drive
{'gpu': 'NVIDIA L4', 'model': 'Qwen/Qwen2.5-3B-Instruct', 'adapter_loaded': False, 'training': False, 'stage4_blocked': True}


In [3]:
"""Deterministic Stage 3.5 zero-shot greenhouse-fan evaluation corpus."""

from collections import Counter
from dataclasses import asdict, dataclass
import itertools
import random
import re
from typing import Sequence


STATES = ("Running", "Stopped")
OPERATIONS = ("same", "different")
DEFAULT_SEED = 20260811
STATE_RE = re.compile(r"^Step (\d+): State: ([A-Z][A-Za-z]{0,14})$", re.MULTILINE)
ANSWER_RE = re.compile(r"<answer>\s*(Running|Stopped)\s*</answer>\s*$", re.IGNORECASE)
RAW_ANSWER_RE = re.compile(r"<answer>\s*([^<\n]+?)\s*</answer>\s*$", re.IGNORECASE)
LATCH_COIN_WORDS = re.compile(
    # "flip" is intentionally not treated as leakage: it is an ordinary
    # domain-neutral verb for a binary transition. The domain-bearing words
    # below would be genuine latch/coin transfer artifacts.
    r"\b(?:latch|locked|unlocked|heads?|tails?|coin)\b", re.IGNORECASE
)
DECLARATION_RE = re.compile(
    r"^Represent Running using the code ([A-Z][A-Za-z]{2,9}) and "
    r"Stopped using the code ([A-Z][A-Za-z]{2,9})\.$",
    re.MULTILINE,
)


@dataclass(frozen=True)
class FanExample:
    example_id: str
    initial_state: str
    operations: tuple[str, ...]
    expected_states: tuple[str, ...]
    final_answer: str
    prompt: str

    def to_dict(self) -> dict[str, object]:
        row = asdict(self)
        row["operations"] = list(self.operations)
        row["expected_states"] = list(self.expected_states)
        return row


@dataclass(frozen=True)
class DeclaredFanExample:
    base: FanExample
    token_for_running: str
    token_for_stopped: str
    prompt: str

    @property
    def example_id(self) -> str:
        return self.base.example_id


def simulate(initial_state: str, operations: Sequence[str]) -> list[str]:
    state = initial_state
    output: list[str] = []
    for operation in operations:
        if operation == "different":
            state = "Stopped" if state == "Running" else "Running"
        elif operation != "same":
            raise ValueError(f"Unknown operation: {operation}")
        output.append(state)
    return output


def _prompt(initial_state: str, operations: Sequence[str]) -> str:
    wording = {
        "same": "same as previously (the fan mode does NOT change)",
        "different": "different from previously (the fan changes to its other mode)",
    }
    lines = [
        f"A greenhouse ventilation fan starts {initial_state}.",
        "Track its mode through every instruction.",
        *[f"{index}. {wording[operation]}" for index, operation in enumerate(operations, 1)],
        "Choose your own two different invented single-word codes for Running and Stopped.",
        "Do not use Running or Stopped in the numbered reasoning lines.",
        "Keep the same code-to-mode mapping throughout this problem.",
        "Write exactly one line per instruction as: Step i: State: <invented code>",
        "End with the actual final fan mode inside <answer>...</answer>.",
    ]
    return "\n".join(lines)


def generate_fan_evaluation(seed: int = DEFAULT_SEED, count: int = 100) -> list[FanExample]:
    rng = random.Random(seed)
    cells: dict[tuple[str, str], list[tuple[str, tuple[str, ...], tuple[str, ...]]]] = {
        (initial, final): [] for initial in STATES for final in STATES
    }
    for length in range(3, 9):
        for initial in STATES:
            for operations in itertools.product(OPERATIONS, repeat=length):
                expected = tuple(simulate(initial, operations))
                if len(set(expected)) != 2:
                    continue
                cells[(initial, expected[-1])].append((initial, operations, expected))
    per_cell = count // 4
    if per_cell * 4 != count:
        raise ValueError("count must be divisible by four")
    selected = []
    for key in sorted(cells):
        rng.shuffle(cells[key])
        selected.extend(cells[key][:per_cell])
    rng.shuffle(selected)
    return [
        FanExample(
            example_id=f"fan-third-domain-{index:03d}",
            initial_state=initial,
            operations=operations,
            expected_states=expected,
            final_answer=expected[-1],
            prompt=_prompt(initial, operations),
        )
        for index, (initial, operations, expected) in enumerate(selected)
    ]


def _nonce_tokens(rng: random.Random):
    consonants, vowels = "bcdfghjklmnprstvwxyz", "aeiou"
    seen: set[str] = set()
    while True:
        token = "".join(rng.choice(consonants) + rng.choice(vowels)
                        for _ in range(rng.choice((2, 3, 4)))).capitalize()
        if token in seen or any(word in token.casefold() for word in
                                ("run", "stop", "lock", "head", "tail", "coin")):
            continue
        seen.add(token)
        yield token


def generate_declared_mapping_control(
    seed: int = DEFAULT_SEED, count: int = 100
) -> list[DeclaredFanExample]:
    """Return the exact third-domain prompts with a unique supplied mapping."""
    base_rows = generate_fan_evaluation(seed, count)
    rng = random.Random(seed + 1)
    tokens = _nonce_tokens(rng)
    output = []
    for index, base in enumerate(base_rows):
        first, second = next(tokens), next(tokens)
        running, stopped = (first, second) if index % 2 == 0 else (second, first)
        declaration = (
            f"Represent Running using the code {running} and Stopped using the code {stopped}."
        )
        prompt = base.prompt.replace(
            "Choose your own two different invented single-word codes for Running and Stopped.",
            declaration,
        ).replace(
            "Keep the same code-to-mode mapping throughout this problem.",
            "Apply this declared code-to-mode mapping throughout this problem.",
        )
        output.append(DeclaredFanExample(base, running, stopped, prompt))
    return output


def verify_declared_control(examples: Sequence[DeclaredFanExample]) -> dict[str, object]:
    failures = []
    for row in examples:
        declarations = DECLARATION_RE.findall(row.prompt)
        if declarations != [(row.token_for_running, row.token_for_stopped)]:
            failures.append(f"{row.example_id}:mapping")
        if LATCH_COIN_WORDS.search(row.prompt):
            failures.append(f"{row.example_id}:leakage")
    return {
        "count": len(examples),
        "unique_prompts": len({row.prompt for row in examples}),
        "unique_token_pairs": len({tuple(sorted((row.token_for_running, row.token_for_stopped)))
                                    for row in examples}),
        "failures": failures,
        "semantic_pass_rate": 100 * (len(examples) - len(failures)) / len(examples),
        "accepted": len(examples) == 100 and not failures,
    }


def score_declared_completion(example: DeclaredFanExample, text: str) -> dict[str, object]:
    score = score_completion(example.base, text)
    expected = {
        "Running": example.token_for_running.casefold(),
        "Stopped": example.token_for_stopped.casefold(),
    }
    score["declared_mapping_adherence"] = (
        bool(score["structural"])
        and score["tokens"] == [expected[state] for state in example.base.expected_states]
    )
    declared_tokens = {
        example.token_for_running.casefold(), example.token_for_stopped.casefold()
    }
    if not score["answer_correct"] and score["raw_answer"] in declared_tokens:
        score["answer_is_code_word"] = True
        score["answer_failure_type"] = "code_word_instead_of_physical_state"
    return score


def verify_corpus(examples: Sequence[FanExample]) -> dict[str, object]:
    failures = []
    for row in examples:
        expected = tuple(simulate(row.initial_state, row.operations))
        if expected != row.expected_states or expected[-1] != row.final_answer:
            failures.append(row.example_id)
        if LATCH_COIN_WORDS.search(row.prompt):
            failures.append(f"{row.example_id}:domain_leakage")
    return {
        "count": len(examples),
        "unique_prompts": len({row.prompt for row in examples}),
        "initial_states": dict(Counter(row.initial_state for row in examples)),
        "final_answers": dict(Counter(row.final_answer for row in examples)),
        "semantic_pass_rate": 100 * (len(examples) - len(failures)) / len(examples),
        "failures": failures,
        "accepted": len(examples) == 100 and not failures,
    }


def score_completion(example: FanExample, text: str) -> dict[str, object]:
    prefix = text.split("<answer>", 1)[0]
    matches = [(int(index), token) for index, token in STATE_RE.findall(prefix)]
    structural = (
        len(matches) == len(example.operations)
        and [index for index, _ in matches] == list(range(1, len(example.operations) + 1))
    )
    tokens = [token.casefold() for _, token in matches]
    literal = {"running", "stopped"}
    leakage = bool(LATCH_COIN_WORDS.search(prefix))
    nonliteral = structural and all(token not in literal for token in tokens) and not leakage
    mapping: dict[str, set[str]] = {state: set() for state in STATES}
    if structural:
        for physical_state, token in zip(example.expected_states, tokens):
            mapping[physical_state].add(token)
    global_consistent = (
        structural
        and all(len(mapping[state]) == 1 for state in STATES)
        and mapping["Running"] != mapping["Stopped"]
    )
    token_pair = None
    if global_consistent:
        token_pair = tuple(sorted((next(iter(mapping["Running"])), next(iter(mapping["Stopped"])))))
    answer = ANSWER_RE.search(text)
    raw_answer_match = RAW_ANSWER_RE.search(text)
    raw_answer = raw_answer_match.group(1).strip().casefold() if raw_answer_match else None
    answer_correct = bool(answer and answer.group(1).casefold() == example.final_answer.casefold())
    answer_is_code_word = bool(
        raw_answer and raw_answer not in {state.casefold() for state in STATES}
        and raw_answer in set(tokens)
    )
    if answer_correct:
        answer_failure_type = None
    elif answer_is_code_word:
        answer_failure_type = "code_word_instead_of_physical_state"
    elif raw_answer is None:
        answer_failure_type = "malformed_or_missing_answer"
    else:
        answer_failure_type = "other_wrong_answer"
    return {
        "structural": structural,
        "nonliteral": nonliteral,
        "global_consistent": global_consistent,
        "nonliteral_consistent": nonliteral and global_consistent,
        "answer_correct": answer_correct,
        "raw_answer": raw_answer,
        "answer_is_code_word": answer_is_code_word,
        "answer_failure_type": answer_failure_type,
        "latch_coin_leakage": leakage,
        "token_pair": token_pair,
        "tokens": tokens,
        "text": text,
    }


In [4]:
examples=generate_declared_mapping_control(SEED,100)
audit=verify_declared_control(examples)
assert audit['accepted'] and audit['semantic_pass_rate']==100.0
assert audit['unique_token_pairs']==100
print('DECLARED-MAPPING CONTROL VERIFIED:',json.dumps(audit,indent=2,sort_keys=True))
print('MATCHED PROMPT EXAMPLE (NO DEMONSTRATION):\n',examples[0].prompt)


DECLARED-MAPPING CONTROL VERIFIED: {
  "accepted": true,
  "count": 100,
  "failures": [],
  "semantic_pass_rate": 100.0,
  "unique_prompts": 100,
  "unique_token_pairs": 100
}
MATCHED PROMPT EXAMPLE (NO DEMONSTRATION):
 A greenhouse ventilation fan starts Running.
Track its mode through every instruction.
1. different from previously (the fan changes to its other mode)
2. different from previously (the fan changes to its other mode)
3. different from previously (the fan changes to its other mode)
4. different from previously (the fan changes to its other mode)
5. same as previously (the fan mode does NOT change)
6. same as previously (the fan mode does NOT change)
7. same as previously (the fan mode does NOT change)
8. same as previously (the fan mode does NOT change)
Represent Running using the code Dizo and Stopped using the code Mecohoyi.
Do not use Running or Stopped in the numbered reasoning lines.
Apply this declared code-to-mode mapping throughout this problem.
Write exactly on

In [5]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code=False)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='left'
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                         bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.bfloat16)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,dtype=torch.bfloat16,quantization_config=quant,device_map={'':0},
    low_cpu_mem_usage=True,use_safetensors=True,trust_remote_code=False)
model.requires_grad_(False); model.eval(); model.config.use_cache=True
assert not any(p.requires_grad for p in model.parameters())
print({'model':MODEL_NAME,'adapter_loaded':False,'trainable_parameters':0,
       'confirmed':'untrained instruction-tuned base model only'})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'model': 'Qwen/Qwen2.5-3B-Instruct', 'adapter_loaded': False, 'trainable_parameters': 0, 'confirmed': 'untrained instruction-tuned base model only'}


In [6]:
SYSTEM='You solve state-tracking tasks accurately and follow the requested output format.'
saved=json.loads(PROGRESS.read_text()) if PROGRESS.is_file() and PROGRESS.stat().st_size else {'rows':[]}
completed={row['example_id'] for row in saved['rows']}
def atomic_json(path,payload):
    temp=path.with_suffix(path.suffix+'.tmp'); temp.write_text(json.dumps(payload,indent=2,sort_keys=True)); temp.replace(path)
for start in range(0,100,2):
    chunk=[row for row in examples[start:start+2] if row.example_id not in completed]
    if not chunk: continue
    prompts=[tokenizer.apply_chat_template(
        [{'role':'system','content':SYSTEM},{'role':'user','content':row.prompt}],
        tokenize=False,add_generation_prompt=True) for row in chunk]
    batch=tokenizer(prompts,return_tensors='pt',padding=True).to(model.device)
    with torch.inference_mode():
        output=model.generate(**batch,max_new_tokens=300,do_sample=False,
                              pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    texts=tokenizer.batch_decode(output[:,batch['input_ids'].shape[1]:],skip_special_tokens=True)
    for example,text in zip(chunk,texts):
        score=score_declared_completion(example,text)
        saved['rows'].append({'example_id':example.example_id,
                              'declared_pair':[example.token_for_running,example.token_for_stopped],
                              'final_answer':example.base.final_answer,**score})
        completed.add(example.example_id)
    atomic_json(PROGRESS,saved); print(f'SAVED CONTROL PROGRESS: {len(completed)}/100')
assert len(saved['rows'])==100 and len(completed)==100


SAVED CONTROL PROGRESS: 2/100
SAVED CONTROL PROGRESS: 4/100
SAVED CONTROL PROGRESS: 6/100
SAVED CONTROL PROGRESS: 8/100
SAVED CONTROL PROGRESS: 10/100
SAVED CONTROL PROGRESS: 12/100
SAVED CONTROL PROGRESS: 14/100
SAVED CONTROL PROGRESS: 16/100
SAVED CONTROL PROGRESS: 18/100
SAVED CONTROL PROGRESS: 20/100
SAVED CONTROL PROGRESS: 22/100
SAVED CONTROL PROGRESS: 24/100
SAVED CONTROL PROGRESS: 26/100
SAVED CONTROL PROGRESS: 28/100
SAVED CONTROL PROGRESS: 30/100
SAVED CONTROL PROGRESS: 32/100
SAVED CONTROL PROGRESS: 34/100
SAVED CONTROL PROGRESS: 36/100
SAVED CONTROL PROGRESS: 38/100
SAVED CONTROL PROGRESS: 40/100
SAVED CONTROL PROGRESS: 42/100
SAVED CONTROL PROGRESS: 44/100
SAVED CONTROL PROGRESS: 46/100
SAVED CONTROL PROGRESS: 48/100
SAVED CONTROL PROGRESS: 50/100
SAVED CONTROL PROGRESS: 52/100
SAVED CONTROL PROGRESS: 54/100
SAVED CONTROL PROGRESS: 56/100
SAVED CONTROL PROGRESS: 58/100
SAVED CONTROL PROGRESS: 60/100
SAVED CONTROL PROGRESS: 62/100
SAVED CONTROL PROGRESS: 64/100
SAVED CONTRO

In [7]:
rows=sorted(saved['rows'],key=lambda row:row['example_id']); n=len(rows)
adherent_pairs=Counter(tuple(sorted(row['declared_pair'])) for row in rows if row['declared_mapping_adherence'])
metrics={
    'evaluation_count':n,
    'structural_rate':sum(row['structural'] for row in rows)/n,
    'nonliteral_encoding_rate':sum(row['nonliteral'] for row in rows)/n,
    'global_mapping_consistency_rate':sum(row['global_consistent'] for row in rows)/n,
    'declared_mapping_adherence_rate':sum(row['declared_mapping_adherence'] for row in rows)/n,
    'final_answer_accuracy':sum(row['answer_correct'] for row in rows)/n,
    'answer_is_code_word_rate':sum(row['answer_is_code_word'] for row in rows)/n,
    'latch_coin_vocabulary_leakage_rate':sum(row['latch_coin_leakage'] for row in rows)/n,
    'distinct_correctly_applied_declared_pairs':len(adherent_pairs),
}
answer_failures=[row for row in rows if not row['answer_correct']]
metrics['answer_is_code_word_fraction_of_failures']=(
    sum(row['answer_is_code_word'] for row in answer_failures)/len(answer_failures)
    if answer_failures else 0.0)
metrics['answer_failure_category_counts']=dict(Counter(
    row['answer_failure_type'] for row in answer_failures))
criteria={
    'mapping_application_at_least_80_percent':metrics['declared_mapping_adherence_rate']>=.80,
    'global_consistency_at_least_80_percent':metrics['global_mapping_consistency_rate']>=.80,
    'answer_accuracy_at_least_80_percent':metrics['final_answer_accuracy']>=.80,
    'structure_at_least_90_percent':metrics['structural_rate']>=.90,
    'nonliteral_at_least_90_percent':metrics['nonliteral_encoding_rate']>=.90,
    'no_domain_leakage':metrics['latch_coin_vocabulary_leakage_rate']==0,
}
report={'stage':'3.5_base_declared_mapping_control','source_model':MODEL_NAME,
        'adapter_loaded':False,'mode':'zero_shot_no_training','audit':audit,
        'metrics':metrics,'criteria':criteria,'passed':all(criteria.values()),'rows':rows,
        'stage4_authorized':False,'checkpoint130_reference':{'declared_mapping_adherence_rate':.05,
                                  'final_answer_accuracy':.01,
                                  'structural_rate':.88},
        'interpretation_thresholds':{'active_damage_if_base_adherence_at_least':.30,
                                     'similar_difficulty_band':[.05,.15]},
        'next_action':'STOP_FOR_REVIEW'}
atomic_json(REPORT,report)
print('===== BASE-MODEL DECLARED-MAPPING CONTROL REPORT =====')
print(json.dumps({**metrics,'criteria':criteria,'passed':report['passed']},indent=2,sort_keys=True))
print('\n===== REPRESENTATIVE RAW COMPLETIONS =====')
for index,row in enumerate(rows[:10],1):
    print(f"\n--- {index}: {row['example_id']} declared={row['declared_pair']} ---\n{row['text']}")
print('\nREPORT SAVED:',REPORT)
print('STOP HERE. No training or Stage 4 composition was performed.')


===== BASE-MODEL DECLARED-MAPPING CONTROL REPORT =====
{
  "answer_failure_category_counts": {
    "code_word_instead_of_physical_state": 85,
    "other_wrong_answer": 12
  },
  "answer_is_code_word_fraction_of_failures": 0.8762886597938144,
  "answer_is_code_word_rate": 0.85,
  "criteria": {
    "answer_accuracy_at_least_80_percent": false,
    "global_consistency_at_least_80_percent": false,
    "mapping_application_at_least_80_percent": false,
    "no_domain_leakage": true,
    "nonliteral_at_least_90_percent": false,
    "structure_at_least_90_percent": false
  },
  "declared_mapping_adherence_rate": 0.03,
  "distinct_correctly_applied_declared_pairs": 3,
  "evaluation_count": 100,
  "final_answer_accuracy": 0.03,
  "global_mapping_consistency_rate": 0.07,
  "latch_coin_vocabulary_leakage_rate": 0.0,
  "nonliteral_encoding_rate": 0.83,
  "passed": false,
  "structural_rate": 0.83
}

===== REPRESENTATIVE RAW COMPLETIONS =====

--- 1: fan-third-domain-000 declared=['Dizo', 'Mecohoyi'